# Tramontane — Mistral-native Agent Orchestration

Route every prompt to the optimal Mistral model. Budget ceilings in EUR. GDPR-native.

Built in Orléans, France 🇫🇷 · [github.com/bleucommerce/tramontane](https://github.com/bleucommerce/tramontane)

In [ ]:
!pip install tramontane -q

## The Router in Action

Tramontane's router classifies your prompt and picks the cheapest Mistral model that can handle it.
No API key needed for the demo — OFFLINE mode uses keyword heuristics.

In [ ]:
from tramontane.router.classifier import ClassificationMode, TaskClassifier
from tramontane.router.router import MistralRouter

classifier = TaskClassifier(mode=ClassificationMode.OFFLINE)
router = MistralRouter()

tasks = [
    "Write a Python function to parse JSON",
    "Analyze the EU AI Act implications for SMEs",
    "Search for the latest news on Mistral AI",
    "List all EU countries",
    "Refactor this monorepo " * 50,
]

for task in tasks:
    d = router.route_sync(task[:200], budget=0.05, locale="fr")
    print(f"{task[:42]:<44} \u2192 {d.primary_model:<20} \u20ac{d.estimated_cost_eur:.4f}")

## Building a Multi-Agent Pipeline

Agents hand off to each other. The router picks the model for each agent independently.

In [ ]:
from tramontane.core.agent import Agent
from tramontane.core.pipeline import Pipeline

researcher = Agent(
    role="Market Researcher",
    goal="Find EU AI market data for 2026",
    backstory="Senior analyst with EU tech market expertise",
    model="auto",
    budget_eur=0.05,
)

analyst = Agent(
    role="Data Analyst",
    goal="Structure findings into actionable insights",
    backstory="Quantitative analyst specializing in market sizing",
    model="auto",
    reasoning=True,
)

pipeline = Pipeline(
    name="market_research",
    agents=[researcher, analyst],
    handoffs=[("Market Researcher", "Data Analyst")],
    budget_eur=0.10,
)

print(f"Pipeline: {pipeline.name}")
print(f"Agents: {list(pipeline.agents.keys())}")
print(f"Graph depth: {pipeline.handoff_graph.depth_from('Market Researcher')}")
print(f"\n{pipeline.handoff_graph.to_mermaid()}")

## GDPR-Native Processing

PII detection runs automatically at `standard` or `strict` GDPR level.
Regex-based OFFLINE mode catches emails, phones, IBAN, French NIR, and more.

In [ ]:
from tramontane.gdpr.pii import PIIDetector

detector = PIIDetector()  # auto OFFLINE when no API key

text = "Contact Jean Dupont at jean.dupont@email.fr or +33 6 12 34 56 78"
result = detector.detect_sync(text)

print(f"PII found: {result.has_pii}")
print(f"Types: {[t.value for t in result.pii_types_found]}")
print(f"Original:  {result.original_text}")
print(f"Redacted:  {result.cleaned_text}")